In [1]:
from langchain_core.tools import tool

In [8]:
import arxiv
from langchain.tools import tool  # optional decorator if using LangChain agents

# Example using LangChain @tool decorator
# @tool
def Arxiv_tool(query: str):
    """
    Use this tool for research paper search on arXiv.
    Returns a list of top 3 papers with title, authors, summary, and PDF link.
    """
    results_list = []

    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )

    for result in search.results():
        paper_info = {
            "title": result.title,
            "authors": [author.name for author in result.authors],
            "summary": result.summary,
            "pdf_url": result.pdf_url,
            "published": result.published.strftime("%Y-%m-%d")
        }
        results_list.append(paper_info)

    return results_list


In [10]:
import arxiv

def Arxiv_tool(query: str):

    results_list = []

    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )

    client = arxiv.Client()

    for result in client.results(search):

        paper_info = {
            "title": result.title,
            "authors": [author.name for author in result.authors],
            "summary": result.summary,
            "pdf_url": result.pdf_url,
            "published": result.published.strftime("%Y-%m-%d")
        }

        results_list.append(paper_info)

    return results_list


print(Arxiv_tool("transformers"))

HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=transformers&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

In [9]:
response = Arxiv_tool("LLms")
# type(response)
response

D:\Temp\ipykernel_28736\2229447718.py:19: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=LLms&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

In [16]:
from langchain_tavily import TavilySearch
Tavily_tool = TavilySearch(
    max_result = 5,
    topic='general'
)

In [17]:
Tavily_tool.invoke("Recent AI news")

{'query': 'Recent AI news',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
   'title': 'Latest AI News and AI Breakthroughs that Matter Most: 2026 & 2025',
   'content': 'Summary: Apple has officially announced that a completely reimagined, AI-powered version of Siri is set to debut in 2026. This fundamental transformation will',
   'score': 0.7971311,
   'raw_content': None},
  {'url': 'https://www.artificialintelligence-news.com/',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'Gartner Data & Analytics Summit unveils expanded AI agenda for 2026 · Thailand becomes one of the first in Asia to get the Sora app · Malaysia launches Ryt Bank',
   'score': 0.7293082,
   'raw_content': None},
  {'url': 'https://blog.google/innovation-and-ai/products/google-ai-updates-january-2026/',
   'title': 'The latest AI news we announced in January - The Key

### WikipediaTool

In [18]:
from langchain_community.tools import WikipediaQueryRun 
from langchain_community.utilities import WikipediaAPIWrapper 
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000)
Wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

## All Tools

In [11]:
import arxiv
from langchain.tools import tool  # optional decorator if using LangChain agents
from langchain_core.tools import Tool
# Arxive Paper Search Tool 
def Arxiv_Search(query: str):
    """
    Use this tool for research paper search on arXiv.
    Returns a list of top 3 papers with title, authors, summary, and PDF link.
    """
    results_list = []

    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )

    for result in search.results():
        paper_info = {
            "title": result.title,
            "authors": [author.name for author in result.authors],
            "summary": result.summary,
            "pdf_url": result.pdf_url,
            "published": result.published.strftime("%Y-%m-%d")
        }
        results_list.append(paper_info)

    return "\n\n".join(
    [f"Title: {p['title']}\nAuthors: {', '.join(p['authors'])}\nSummary: {p['summary'][:300]}...\nPDF: {p['pdf_url']}\nPublished: {p['published']}" 
     for p in results_list]
)
Arxiv_tool = Tool(
    name="ArXiv_Search",
    func=Arxiv_Search,
    description="Search for research papers on arXiv using a query and get top results."
)

# Tavily Web Search tool 
def Tavily_tool_func(query: str):
    # call TavilySearch here
    return TavilySearch(max_result=5, topic='general').run(query)

Tavily_tool = Tool(
    name="Tavily_Search",
    func=Tavily_tool_func,
    description="Search the web for general information"
)


# Wikipedia Search 
wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000)

Wiki_tool = Tool(
    name="Wikipedia_Search",
    func=wiki.run,
    description="Use this tool to get information from Wikipedia"
)

NameError: name 'WikipediaAPIWrapper' is not defined

In [59]:
Tavily_tool.name,Wiki_tool.name,Arxiv_tool.name

('Tavily_Search', 'Wikipedia_Search', 'ArXiv_Search')

In [60]:
tools = [Arxiv_tool,Tavily_tool,Wiki_tool] 

In [61]:
for tool in tools:
    print(type(tool))

<class 'langchain_core.tools.simple.Tool'>
<class 'langchain_core.tools.simple.Tool'>
<class 'langchain_core.tools.simple.Tool'>


In [62]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='llama-3.3-70b-versatile',temperature=0)

In [63]:
LLM_with_tools = llm.bind_tools(tools)

In [80]:
response = LLM_with_tools.invoke("LLms research paper")

In [82]:
response.tool_calls

[{'name': 'ArXiv_Search',
  'args': {'__arg1': 'LLMs research paper'},
  'id': 'ckv2y86w8',
  'type': 'tool_call'}]